# Food Waste Estimation -- Training Pipeline

10-fold cross-validation training of the dual-stream EfficientNet-B0 model.
Runs on local (CPU/GPU) and Google Colab Pro (T4 GPU).

**Run cells top to bottom. Do not skip the setup cell.**

## 0. Environment Setup

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/food-waste-estimation'  # adjust if needed
    os.chdir(PROJECT_ROOT)
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    PROJECT_ROOT = os.getcwd()

sys.path.insert(0, 'src')
print(f'{"Colab" if IN_COLAB else "Local"} | cwd: {os.getcwd()}')

## 1. Install Dependencies

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-r', 'requirements.txt'], check=True)
    print('Dependencies installed.')
else:
    print('Local: run "uv sync" in terminal if not done yet.')

## 2. Imports and Config

In [ ]:
%matplotlib inline
import glob
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold

from utils import set_seed, compute_mae, compute_rmse
from dataset import FoodWasteDataset, load_metadata, get_transforms
from model import DualStreamEfficientNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training config -- edit here
CONFIG = {
    'folds':       10,
    'epochs':      100,
    'lr':          1e-4,
    'batch_size':  16,
    'patience':    20,
    'seed':        42,
    'num_workers': 2,
    'pin_memory':  DEVICE.type == 'cuda',
}

# Paths
DATA_DIR       = 'data'
BEFORE_DIR     = os.path.join(DATA_DIR, 'segmented', 'data_before')
AFTER_DIR      = os.path.join(DATA_DIR, 'segmented', 'data_after')
CHECKPOINT_DIR = 'checkpoints'
RESULTS_DIR    = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Config: {CONFIG}')

## 3. Load and Filter Metadata

In [ ]:
set_seed(CONFIG['seed'])

df, norm_params, le = load_metadata(
    os.path.join(DATA_DIR, 'data_original.xlsx'),
    before_dir=BEFORE_DIR,
    after_dir=AFTER_DIR,
    save_dir=RESULTS_DIR
)

label_classes = le.classes_.tolist()
num_classes   = len(label_classes)

print(f'Usable samples : {len(df)}')
print(f'Categories     : {num_classes}')
print(f'Max weight     : {norm_params["max_weight"]}g')
df.head()

## 4. Training

In [ ]:
train_transform = get_transforms('train')
val_transform   = get_transforms('val')

skf    = StratifiedKFold(n_splits=CONFIG['folds'], shuffle=True, random_state=CONFIG['seed'])
labels = df['category_id'].values

fold_indices = {}
fold_maes    = []
fold_logs    = {}

mse_loss = nn.MSELoss()
ce_loss  = nn.CrossEntropyLoss()

for fold_n, (trainval_idx, test_idx) in enumerate(skf.split(np.arange(len(df)), labels), start=1):
    print(f'\n{"="*60}')
    print(f'Fold {fold_n}/{CONFIG["folds"]}')
    print(f'{"="*60}')

    set_seed(CONFIG['seed'] + fold_n)

    n_val      = int(len(trainval_idx) * (0.2 / 0.9))
    np.random.shuffle(trainval_idx)
    val_idx    = trainval_idx[:n_val]
    train_idx  = trainval_idx[n_val:]

    fold_indices[f'fold_{fold_n}'] = {
        'train': train_idx.tolist(),
        'val':   val_idx.tolist(),
        'test':  test_idx.tolist()
    }

    train_ds = FoodWasteDataset(df.iloc[train_idx], BEFORE_DIR, AFTER_DIR, train_transform)
    val_ds   = FoodWasteDataset(df.iloc[val_idx],   BEFORE_DIR, AFTER_DIR, val_transform)

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                              num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,
                              num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'])

    model     = DualStreamEfficientNet(num_classes=num_classes, pretrained=True).to(DEVICE)
    model.freeze_backbone()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])

    best_val_mae      = float('inf')
    epochs_no_improve = 0
    log_rows          = []

    for epoch in range(1, CONFIG['epochs'] + 1):
        if epoch == 6:
            model.unfreeze_backbone()

        model.train()
        train_losses = []
        for batch in train_loader:
            before        = batch['before'].to(DEVICE)
            after         = batch['after'].to(DEVICE)
            leftover_norm = batch['leftover_norm'].to(DEVICE)
            category      = batch['category'].to(DEVICE)

            optimizer.zero_grad()
            pred_left, pred_cat = model(before, after)
            loss = 0.9 * mse_loss(pred_left, leftover_norm) + 0.1 * ce_loss(pred_cat, category)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_preds, val_targets, val_cat_preds, val_cat_targets, val_losses = [], [], [], [], []

        with torch.no_grad():
            for batch in val_loader:
                before        = batch['before'].to(DEVICE)
                after         = batch['after'].to(DEVICE)
                leftover_norm = batch['leftover_norm'].to(DEVICE)
                category      = batch['category'].to(DEVICE)

                pred_left, pred_cat = model(before, after)
                loss = 0.9 * mse_loss(pred_left, leftover_norm) + 0.1 * ce_loss(pred_cat, category)
                val_losses.append(loss.item())
                val_preds.append(pred_left.cpu())
                val_targets.append(leftover_norm.cpu())
                val_cat_preds.append(pred_cat.argmax(dim=1).cpu())
                val_cat_targets.append(category.cpu())

        val_preds       = torch.cat(val_preds)
        val_targets     = torch.cat(val_targets)
        val_cat_preds   = torch.cat(val_cat_preds)
        val_cat_targets = torch.cat(val_cat_targets)

        val_mae  = compute_mae(val_preds, val_targets)
        val_rmse = compute_rmse(val_preds, val_targets)
        val_acc  = float((val_cat_preds == val_cat_targets).float().mean())

        print(f'  Epoch {epoch:3d} | '
              f'Train Loss: {np.mean(train_losses):.4f} | '
              f'Val Loss: {np.mean(val_losses):.4f} | '
              f'Val MAE: {val_mae:.4f} | '
              f'Val Acc: {val_acc:.4f}')

        log_rows.append({
            'epoch': epoch, 'train_loss': float(np.mean(train_losses)),
            'val_loss': float(np.mean(val_losses)),
            'val_mae': val_mae, 'val_rmse': val_rmse, 'val_acc': val_acc
        })

        if val_mae < best_val_mae:
            best_val_mae      = val_mae
            epochs_no_improve = 0
            os.makedirs(CHECKPOINT_DIR, exist_ok=True)
            torch.save({
                'fold': fold_n, 'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_mae': best_val_mae,
                'normalization_params': norm_params,
                'label_encoder_classes': label_classes
            }, os.path.join(CHECKPOINT_DIR, f'fold_{fold_n}_best.pth'))
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CONFIG['patience']:
                print(f'  Early stopping at epoch {epoch}')
                break

    fold_maes.append(best_val_mae)
    fold_logs[fold_n] = pd.DataFrame(log_rows)
    fold_logs[fold_n].to_csv(os.path.join(RESULTS_DIR, f'fold_{fold_n}_log.csv'), index=False)
    print(f'Fold {fold_n} best val MAE: {best_val_mae:.4f}')

with open(os.path.join(RESULTS_DIR, 'fold_indices.json'), 'w') as f:
    json.dump(fold_indices, f, indent=2)

print(f'\nAll folds complete.')

## 5. Results Summary

In [ ]:
# Recover fold_maes and fold_logs from disk if the kernel was restarted after training.
try:
    fold_maes
except NameError:
    with open(os.path.join(RESULTS_DIR, 'summary.json')) as _f:
        _s = json.load(_f)
    fold_maes = _s['fold_maes']
    print('Loaded fold_maes from summary.json')

try:
    fold_logs
except NameError:
    fold_logs = {}
    for _csv in sorted(glob.glob(os.path.join(RESULTS_DIR, 'fold_*_log.csv'))):
        _n = int(os.path.basename(_csv).split('_')[1])
        fold_logs[_n] = pd.read_csv(_csv)
    print(f'Loaded {len(fold_logs)} fold logs from disk')

summary = {
    'fold_maes':          fold_maes,
    'mean_mae':           float(np.mean(fold_maes)),
    'std_mae':            float(np.std(fold_maes)),
    'human_baseline_mae': 0.0926,
    'beats_baseline':     float(np.mean(fold_maes)) < 0.0926
}
with open(os.path.join(RESULTS_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Mean MAE : {summary["mean_mae"]:.4f} +/- {summary["std_mae"]:.4f}')
print(f'Baseline : 0.0926  |  Beats baseline: {summary["beats_baseline"]}')

# Per-fold MAE bar chart
plt.figure(figsize=(10, 4))
plt.bar(range(1, len(fold_maes) + 1), fold_maes, color='steelblue')
plt.axhline(0.0926, color='red', linestyle='--', label='Human baseline (0.0926)')
plt.axhline(summary['mean_mae'], color='green', linestyle='--',
            label=f'Mean MAE ({summary["mean_mae"]:.4f})')
plt.xlabel('Fold')
plt.ylabel('Val MAE (normalized)')
plt.title('Per-Fold Validation MAE')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fold_mae.png'), dpi=120)
plt.show()

# Training curves for each fold
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True)
for fold_n, ax in enumerate(axes.flat, start=1):
    if fold_n in fold_logs:
        log = fold_logs[fold_n]
        ax.plot(log['epoch'], log['val_mae'], label='Val MAE')
        ax.axhline(0.0926, color='red', linestyle='--', linewidth=0.8)
        ax.set_title(f'Fold {fold_n}')
        ax.set_xlabel('Epoch')
plt.suptitle('Val MAE per Epoch per Fold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=120)
plt.show()